In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [ ]:
# Mendefinisikan korpus atau kumpulan dokumen teks dengan topik Kuliner Nusantara
kumpulan_dokumen = [
    "Nasi goreng adalah makanan khas Nusantara yang sangat gurih.",
    "Sate ayam disajikan dengan bumbu kacang yang kental dan lezat.",
    "Rendang daging sapi dimasak perlahan menggunakan santan kelapa.",
    "Gado-gado merupakan campuran sayuran segar dengan siraman bumbu kacang."
]

In [ ]:
# Tahapan pre-processing mencakup: case folding (huruf kecil), menghilangkan selain abjad (regex), dan menghapus stopword
penghapus_stopword = StopWordRemoverFactory().get_stop_words()

def bersihkan_dokumen(kalimat):
    teks_kecil = kalimat.lower()
    hanya_huruf = re.sub(r'[^a-z\s]', ' ', teks_kecil)
    kata_terpilih = [k for k in hanya_huruf.split() if k not in penghapus_stopword]
    return ' '.join(kata_terpilih)

dokumen_bersih = [bersihkan_dokumen(dok) for dok in kumpulan_dokumen]
print("Hasil Pembersihan Teks:\n", dokumen_bersih, "\n")

In [ ]:
# Menyusun dan mengurutkan seluruh kosa kata unik dari dokumen yang telah bersih (pembentukan vocabulary)
semua_kata = sorted(list(set(' '.join(dokumen_bersih).split())))
total_dok = len(dokumen_bersih)
indeks_dok = [f"Dok-{i+1}" for i in range(total_dok)]

frekuensi_kata = [{kata: dok.split().count(kata) for kata in semua_kata} for dok in dokumen_bersih]
df_tf = pd.DataFrame(frekuensi_kata, index=indeks_dok)

df_idf = np.log((1 + total_dok) / (1 + (df_tf > 0).sum(axis=0))) + 1
df_tfidf_raw = df_tf * df_idf
df_tfidf_final = df_tfidf_raw.div(np.linalg.norm(df_tfidf_raw, axis=1), axis=0)

print("\n--- PERHITUNGAN TF-IDF MANUAL ---")
display(df_tfidf_final)

In [ ]:
# Menampilkan hasil perhitungan otomatis dan membandingkannya (selisih) dengan hasil hitung manual
vectorizer = TfidfVectorizer(vocabulary=semua_kata)
tfidf_matriks = vectorizer.fit_transform(dokumen_bersih)
df_tfidf_sklearn = pd.DataFrame(tfidf_matriks.toarray(), columns=semua_kata, index=indeks_dok)

print("\n--- PERHITUNGAN TF-IDF SKLEARN ---")
display(df_tfidf_sklearn)

selisih = (df_tfidf_final - df_tfidf_sklearn).abs().max().max()
print(f"\nTingkat Error (Manual vs Sklearn): {selisih}")

**Kesimpulan TF-IDF:**
Berdasarkan penerapan metode TF-IDF pada korpus teks kuliner di atas, sistem berhasil memberikan bobot lebih tinggi pada kata-kata spesifik (seperti 'sate', 'rendang', 'gado') dibandingkan kata yang sering muncul di berbagai dokumen (seperti 'kacang' atau 'bumbu'). Hal ini menunjukkan bahwa TF-IDF sangat efektif dalam menonjolkan fitur unik dari setiap dokumen. Jika diimplementasikan dalam mesin pencari, hasil pencarian akan jauh lebih akurat dan relevan dengan kata kunci dari pengguna.